In [21]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

## ViT

In [ ]:
# CIFAR-10 normalizasyon değerleri
mean = (0.4914, 0.4822, 0.4465)
std  = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])
data_path = r"C:\Users\oguzh\OneDrive\Masaüstü\data\CIFAR-10"
train_dataset = datasets.CIFAR10(
    root=data_path,
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = datasets.CIFAR10(
    root=data_path,
    train=False,
    download=True,
    transform=test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

0.4%

In [18]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self,embed_dim=64, num_heads=8, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attention = nn.MultiheadAttention(embed_dim= embed_dim, num_heads= num_heads, batch_first= True)
        self.norm2 = nn.LayerNorm(embed_dim)
        hidden_dim = embed_dim * mlp_ratio

        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embed_dim)
        )

    def forward(self, X):
        z = self.norm1(X)
        attention, _ = self.attention(z,z,z, need_weights= False)
        X += attention

        X += self.mlp(self.norm2(X))

        return X


In [19]:
class ViT(nn.Module):
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        in_channels=3,
        num_classes=10,
        embed_dim=64,
        num_heads=8,
        depth=6
    ):
        super().__init__()
        assert img_size % patch_size == 0
        assert embed_dim % num_heads == 0

        num_patches = (img_size // patch_size) ** 2

        # 1. Patch embedding
        self.patch_embedding = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

        # 2. CLS ve positional embedding
        self.cls_token = nn.Parameter(
            torch.zeros(1, 1, embed_dim)
        )
        self.pos_embedding = nn.Parameter(
            torch.zeros(1, num_patches + 1, embed_dim)
        )

        # Küçük, öğrenilebilir başlangıç değerleri
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embedding, std=0.02)

        # 3. Transformer Encoder blokları
        self.blocks = nn.Sequential(*[
            TransformerEncoderBlock(embed_dim, num_heads)
            for _ in range(depth)
        ])

        # 4. Son normalizasyon ve sınıflandırıcı
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.shape[0]

        # (B, 3, 224, 224) -> (B, 64, 14, 14)
        x = self.patch_embedding(x)

        # (B, 64, 14, 14) -> (B, 196, 64)
        x = x.flatten(2).transpose(1, 2)

        # CLS: (B, 1, 64)
        cls = self.cls_token.expand(B, -1, -1)

        # (B, 196, 64) -> (B, 197, 64)
        x = torch.cat([cls, x], dim=1)

        # Konum bilgisini ekle
        x = x + self.pos_embedding

        # Transformer Encoder
        x = self.blocks(x)

        # CLS token'ını seç: (B, 197, 64) -> (B, 64)
        x = self.norm(x[:, 0, :])

        # Sınıf skorları: (B, 64) -> (B, num_classes)
        x = self.head(x)

        return x

In [20]:
model = ViT(
    img_size=224,
    patch_size=16,
    num_classes=10,
    embed_dim=64,
    num_heads=8,
    depth=6
)

images = torch.randn(4, 3, 224, 224)

logits = model(images)

print(logits.shape)

torch.Size([4, 10])
